# 动手实验：基于 PyTorch 的理性期权行权与借款人行为神经网络对比

本实战笔记本对应《AI 时代的房贷市场》第五章【房贷中的期权属性】——【机器学习桥梁：理性选择与行为习惯的神经网络建模】。

## 金融业务背景：为什么房贷借款人不是理性博弈棋手？
在经典的金融衍生品理论（Black-Scholes-Merton）中，期权持有者被假定为绝对理性的决策者：
- 美式期权在标的资产价格跨越最优停时边界（Smooth Pasting 边界）的瞬间，会以 100% 的确定性被立即行权；
- 从结构上看，杰克持有的住房抵押贷款所附带的提前还款权，宛如一份以剩余本金为行权价的美式看涨期权。

然而在现实房贷市场中，借款人行为与纯粹的理性金融期权存在本质裂痕：
1. **核保信用阻尼（Underwriting Friction）**：哪怕市场利率下行 200 个基点，如果借款人的信用分跌破 620 或因房价下跌陷入负资产（LTV > 80%），商业银行严格的核保门槛会直接将其拒之门外；
2. **金额尺度非对称（Scale Asymmetry）**：50 个基点的利差激励在 50 万美元的大额房贷上每年可节省 2,500 美元利息，足以覆盖手续费摩擦；而在 8 万美元的小额存量贷款上每年仅节省 400 美元，绝大多数借款人会选择忽视；
3. **疲劳损耗效应（Burnout & Cognitive Inertia）**：在历次降息周期中数次未能再融资的借款人群体，呈现出系统性的行为惰性。

## 行为神经网络（Behavioral Neural Network）范式
现代机器学习不再依赖人为硬编码的单因子 S 曲线或刀锋般的阶跃边界，而是直接在全维度贷款数据上拟合多层**行为分类神经网络**：
$$p = \sigma(f_\theta(x, \text{incentive})) = \frac{1}{1 + \exp(-f_\theta(x, \text{incentive}))}$$
其中 $x = [\text{利差激励}, \text{FICO 信用分}, \text{LTV 资产负债率}, \text{贷款余额}, \text{损耗计数}]$。

## 本实战涵盖的核心模块
1. 模拟 10,000 笔包含真实微观异质性与行为摩擦的房贷数据集；
2. 实现经典理性美式期权阶跃模型与单因子经验 S 曲线作为行业基准；
3. 构建基于 PyTorch 的深度行为分类器 `BehavioralNet` 并完成端到端训练；
4. 实证对比不同借款人画像的行为差异：优质杰克 vs 受阻鲍勃；
5. 绘制（利差激励 vs 信用分）二维连续行权概率热力图；
6. 在样本外测试集上评估 ROC-AUC 判别能力与相对增益。


In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

# 限制 CPU 线程为 4，避免多核环境下的 OpenMP 锁竞争
torch.set_num_threads(4)

# 设置随机种子以保证实验完全可复现
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# 配置 matplotlib 支持中文字体显示
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'WenQuanYi Zen Hei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("PyTorch 版本:", torch.__version__)
print("当前使用的 CPU 线程数:", torch.get_num_threads())


PyTorch 版本: 2.10.0+cu128
当前使用的 CPU 线程数: 4


## 第一步：生成包含微观行为异质性的 10,000 笔合成贷款

数据涵盖驱动借款人决策的核心维度：
- **利差激励（Incentive）**：原票面利率减去当前市场再融资利率（$-100$ 至 $+300$ bp）；
- **FICO 信用分**：$580$ 至 $820$；
- **LTV 资产负债率**：$40\%$ 至 $105\%$；
- **贷款本金余额**：$5 万至 $65 万美元；
- **疲劳损耗计数（Burnout）**：过往降息潮中错过的再融资次数（$0$ 至 $5$ 次）。


In [2]:
N_SAMPLES = 10000

def generate_cohort(n_samples=N_SAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    
    incentive = rng.uniform(-100.0, 300.0, size=n_samples)
    fico = rng.normal(720.0, 50.0, size=n_samples).clip(580, 820)
    ltv = rng.normal(72.0, 12.0, size=n_samples).clip(40, 105)
    balance = rng.lognormal(mean=12.2, sigma=0.45, size=n_samples).clip(50000, 650000)
    burnout = rng.poisson(lam=0.8, size=n_samples).clip(0, 5)

    X = np.column_stack([incentive, fico, ltv, balance, burnout])

    # 真值：摩擦只作用于再融资响应（正的激励部分）
    inc = incentive / 100.0
    credit_gate = 1.0 / (1.0 + np.exp(-(fico - 670.0) / 25.0))
    equity_gate = 1.0 / (1.0 + np.exp((ltv - 82.0) / 5.0))
    scale = 0.8 + 0.5 * np.log(balance / 50000.0) / np.log(10.0)
    refi = 2.2 * np.maximum(inc, 0.0) * credit_gate * equity_gate * scale
    lock_in = 0.8 * np.minimum(inc, 0.0)
    true_logit = -2.5 + lock_in + refi - 0.45 * burnout
    true_prob = 1.0 / (1.0 + np.exp(-true_logit))
    true_prob = true_prob.clip(0.005, 0.95)

    y = (rng.uniform(0.0, 1.0, size=n_samples) < true_prob).astype(np.float32)
    return X, y, true_prob

X, y, true_prob = generate_cohort()
print(f"成功生成 {len(X)} 笔贷款样本。")
print(f"经验提前还款发生率: {y.mean()*100:.2f}%")
print(f"样本均值: FICO={X[:, 1].mean():.1f}, LTV={X[:, 2].mean():.1f}%, 贷款余额=${X[:, 3].mean():,.0f}")


成功生成 10000 笔贷款样本。
经验提前还款发生率: 27.02%
样本均值: FICO=720.3, LTV=72.3%, 贷款余额=$219,140


## 第二步：经典行权基准模型

建立两类经典模型作为对比基准：
1. **理性美式期权阶跃模型**：若利差激励大于覆盖交易成本的门槛（50 bp），则 100% 触发还款；
2. **单因子经验 S 曲线**：仅依赖利差单一特征的传统 Logistic 曲线。


In [3]:
# 1. 理性阶跃模型 (利差激励 >= 50 bp 立即行权)
score_rational = (X[:, 0] >= 50.0).astype(float)

# 2. 单因子经验 S 曲线 (仅基于利差单一变量)
score_heuristic = 1.0 / (1.0 + np.exp(-(X[:, 0] - 80.0) / 45.0))

auc_rat = roc_auc_score(y, score_rational)
auc_heu = roc_auc_score(y, score_heuristic)

print(f"理性期权阶跃模型基准 AUC: {auc_rat:.4f}")
print(f"传统单因子经验 S 曲线 AUC: {auc_heu:.4f}")


理性期权阶跃模型基准 AUC: 0.7099
传统单因子经验 S 曲线 AUC: 0.8182


## 第三步：构建 PyTorch 行为神经网络模型架构

构建包含 Dropout 正则化与 Sigmoid 输出层的前馈多层感知机，用于直接输出连续行权概率 $p \in [0, 1]$。


In [4]:
class BehavioralNet(nn.Module):
    """预测房贷提前还款行为概率的深度网络"""

    def __init__(self, in_features: int = 5, hidden_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

model = BehavioralNet(in_features=5, hidden_dim=32)
print(model)
print(f"模型可训练参数总量: {sum(p.numel() for p in model.parameters())}")


BehavioralNet(
  (net): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
    (6): Sigmoid()
  )
)
模型可训练参数总量: 1281


## 第四步：模型训练与验证

按照 80/20 划分训练集与测试集，利用二元交叉熵损失（BCE Loss）与 Adam 优化器完成训练。


In [5]:
# 80/20 训练集与测试集切分
n_total = len(X)
perm = np.random.permutation(n_total)
n_tr = int(0.8 * n_total)
train_idx, test_idx = perm[:n_tr], perm[n_tr:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# 特征标准化
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-7

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

X_tr_t = torch.tensor(X_train_norm, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32)
X_te_t = torch.tensor(X_test_norm, dtype=torch.float32)
y_te_t = torch.tensor(y_test, dtype=torch.float32)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

epochs = 150
history = {"loss": [], "auc": []}

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    preds = model(X_tr_t)
    loss = criterion(preds, y_tr_t)
    loss.backward()
    optimizer.step()
    
    if epoch % 25 == 0 or epoch == epochs:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_te_t).numpy()
            test_auc = roc_auc_score(y_test, test_preds)
        history["loss"].append(loss.item())
        history["auc"].append(test_auc)
        print(f"轮次 {epoch:3d}/{epochs} | 训练损失: {loss.item():.4f} | 测试集 AUC: {test_auc:.4f}")


轮次  25/150 | 训练损失: 0.3983 | 测试集 AUC: 0.8627
轮次  50/150 | 训练损失: 0.3769 | 测试集 AUC: 0.8685
轮次  75/150 | 训练损失: 0.3719 | 测试集 AUC: 0.8679
轮次 100/150 | 训练损失: 0.3682 | 测试集 AUC: 0.8683
轮次 125/150 | 训练损失: 0.3631 | 测试集 AUC: 0.8679
轮次 150/150 | 训练损失: 0.3637 | 测试集 AUC: 0.8679


## 第五步：借款人微观画像对比：优质杰克 vs 受阻鲍勃

通过控制变量法，对比两类典型借款人在相同利差激励下的行权概率曲线：
- **优质借款人（杰克）**：FICO 790，LTV 65%，贷款余额 45 万美元，损耗 0（资质优异，金融敏感）；
- **受阻借款人（鲍勃）**：FICO 640，LTV 88%，贷款余额 15 万美元，损耗 2（信用门槛受限，金额吸引力低）。


In [6]:
incentives = np.linspace(-100, 300, 200)

x_prime = np.column_stack([
    incentives,
    np.full_like(incentives, 790.0),
    np.full_like(incentives, 65.0),
    np.full_like(incentives, 400000.0),
    np.full_like(incentives, 0.0),
])

x_frictional = np.column_stack([
    incentives,
    np.full_like(incentives, 640.0),
    np.full_like(incentives, 88.0),
    np.full_like(incentives, 150000.0),
    np.full_like(incentives, 2.0),
])

x_prime_norm = torch.tensor((x_prime - mean) / std, dtype=torch.float32)
x_fric_norm = torch.tensor((x_frictional - mean) / std, dtype=torch.float32)

model.eval()
with torch.no_grad():
    p_prime = model(x_prime_norm).numpy()
    p_frictional = model(x_fric_norm).numpy()

p_rational = (incentives >= 50.0).astype(float)
p_heuristic = 1.0 / (1.0 + np.exp(-(incentives - 80.0) / 45.0))

print("当再融资利差为 +150 bp 时：")
idx_150 = np.argmin(np.abs(incentives - 150.0))
print(f"  优质杰克预测行权还款概率:       {p_prime[idx_150]*100:.1f}%")
print(f"  受阻鲍勃预测行权还款概率:       {p_frictional[idx_150]*100:.1f}%")
print(f"  传统单因子 S 曲线的无差别预测:  {p_heuristic[idx_150]*100:.1f}%")


当再融资利差为 +150 bp 时：
  优质杰克预测行权还款概率:       78.9%
  受阻鲍勃预测行权还款概率:       5.4%
  传统单因子 S 曲线的无差别预测:  82.3%


## 第六步：量化研究全景对照图表

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

# 子图 1: 借款人画像行权曲线
ax1 = axes[0]
ax1.plot(incentives, p_rational, "k--", lw=1.8, label="理性美式期权阶跃边界")
ax1.plot(incentives, p_heuristic, color="gray", ls="-.", lw=2.0, label="传统单因子 S 曲线")
ax1.plot(incentives, p_prime, color="#1b4965", lw=2.4, label="高信用大额借款人 (杰克)")
ax1.plot(incentives, p_frictional, color="#e63946", lw=2.4, label="低信用受阻借款人 (鲍勃)")
ax1.axvline(50, color="gray", ls=":", lw=1.0)
ax1.set_xlabel("再融资利差激励 (bp)", fontsize=11)
ax1.set_ylabel("行权还款概率 $p$", fontsize=11)
ax1.set_title("理性行权 vs 借款人行为概率曲线", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.3, ls="--")
ax1.legend(loc="upper left", framealpha=0.9, fontsize=9)

# 子图 2: 二维行权概率热力图 (利差 vs 信用)
inc_mesh = np.linspace(-50, 250, 60)
fico_mesh = np.linspace(600, 820, 60)
INC, FICO = np.meshgrid(inc_mesh, fico_mesh)
grid_flat = np.column_stack([
    INC.flatten(),
    FICO.flatten(),
    np.full(INC.size, 72.0),
    np.full(INC.size, 300000.0),
    np.full(INC.size, 0.0),
])
grid_norm = torch.tensor((grid_flat - mean) / std, dtype=torch.float32)
with torch.no_grad():
    Z = model(grid_norm).numpy().reshape(INC.shape)

ax2 = axes[1]
c = ax2.contourf(INC, FICO, Z, levels=20, cmap="viridis")
cbar = fig.colorbar(c, ax=ax2)
cbar.set_label("提前还款概率 $p$", fontsize=10)
ax2.set_xlabel("利差激励 (bp)", fontsize=11)
ax2.set_ylabel("FICO 信用分", fontsize=11)
ax2.set_title("行为行权概率热力图", fontsize=12, fontweight="bold")

# 子图 3: 样本外 ROC 曲线对比
ax3 = axes[2]
X_eval, y_eval, _ = generate_cohort(n_samples=3000, seed=999)
X_eval_norm = torch.tensor((X_eval - mean) / std, dtype=torch.float32)
with torch.no_grad():
    score_nn_eval = model(X_eval_norm).numpy()
score_heu_eval = 1.0 / (1.0 + np.exp(-(X_eval[:, 0] - 80.0) / 45.0))
score_rat_eval = (X_eval[:, 0] >= 50.0).astype(float)

auc_nn_val = roc_auc_score(y_eval, score_nn_eval)
auc_heu_val = roc_auc_score(y_eval, score_heu_eval)
auc_rat_val = roc_auc_score(y_eval, score_rat_eval)

for sc, lbl, col in [
    (score_nn_eval, f"行为神经网络 (AUC = {auc_nn_val:.3f})", "#1b4965"),
    (score_heu_eval, f"传统单因子 S 曲线 (AUC = {auc_heu_val:.3f})", "#2a9d8f"),
    (score_rat_eval, f"理性阶跃模型 (AUC = {auc_rat_val:.3f})", "#e76f51"),
]:
    thresholds = np.linspace(0, 1, 100)
    tpr, fpr = [], []
    for th in thresholds:
        yp = (sc >= th).astype(int)
        tp = np.sum((yp == 1) & (y_eval == 1))
        fp = np.sum((yp == 1) & (y_eval == 0))
        fn = np.sum((yp == 0) & (y_eval == 1))
        tn = np.sum((yp == 0) & (y_eval == 0))
        tpr.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    ax3.plot(fpr, tpr, color=col, lw=2.2, label=lbl)

ax3.plot([0, 1], [0, 1], "k:", lw=1.0)
ax3.set_xlabel("假正率 (FPR)", fontsize=11)
ax3.set_ylabel("真正率 (TPR)", fontsize=11)
ax3.set_title("ROC 判别能力基准测试", fontsize=12, fontweight="bold")
ax3.grid(True, alpha=0.3, ls="--")
ax3.legend(loc="lower right", framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()


/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 20877 (\N{CJK UNIFIED IDEOGRAPH-518D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 34701 (\N{CJK UNIFIED IDEOGRAPH-878D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 36164 (\N{CJK UNIFIED IDEOGRAPH-8D44}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 21033 (\N{CJK UNIFIED IDEOGRAPH-5229}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 24046 (\N{CJK UNIFIED IDEOGRAPH-5DEE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 28608 (\N{CJK UNIFIED IDEOGRAPH-6FC0}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_336137/3623785645.py:76: UserWarning: Glyph 21169 (\N{CJK UNIFIED I

Font 'default' does not have a glyph for '\u884c' [U+884c], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u6743' [U+6743], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u8fd8' [U+8fd8], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u6b3e' [U+6b3e], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u6982' [U+6982], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u7387' [U+7387], substituting with a dummy symbol.
/home/wukong/miniconda3/envs/usao/lib/python3.12/site-packages/IPython/core/pylabtools.py:170: UserWarning: Glyph 29702 (\N{CJK UNIFIED IDEOGRAPH-7406}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/home/wukong/miniconda3/envs/usao/lib/python3.12/site-packages/IPython/core/pylabtools.py:170: UserWarning: Glyph 24615 (\N{CJK UNIFIED IDEOGRAPH-6027}) missing from font(s) DejaVu Sans.
  fig.canvas.print_f

## 量化要点与核心启示

1. **破除纯理性行权幻想**：房贷借款人不是金融期权的冰冷行权机器。信用门槛、房屋净值、贷款金额与历史损耗，使得在完全相同的利率利差下，不同画像借款人的行权概率产生巨大分化。
2. **超越单因子 S 曲线**：传统的一维 S 曲线将所有具有相同利差的贷款等同视之。行为神经网络通过多维特征协同建模，将测试集 ROC-AUC 从 $0.818$ 提升至 $0.868$（提高 0.05）。在 +150 bp 的激励下，它给优质的杰克约 79% 的行权概率，给受阻的鲍勃约 5%，而 S 曲线给两人的都是同一个 82%。
3. **交易台落地与竞争风险**：在量化交易台的实战系统（如第 7 章与第 9 章）中，该行为神经网络分类器构成了“提前还款、违约、持续在贷”多状态转移矩阵与混合残差模型的核心驱动引擎。
